In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

# ── Global constants ──────────────────────────────────────────────────────────
N_SPECIES    = 1000          # 1 pathogen + community
N_RESOURCES  = 30
N_GENES      = 10
N_BROTHS     = 30

PATHOGEN_IDX  = 0
N_COMMUNITY   = N_SPECIES - 1
N_SPECIALIST  = N_COMMUNITY // 2          # ~333 specialists
N_GENERALIST  = N_COMMUNITY - N_SPECIALIST  # ~666 generalists

# Partial community: how many non-pathogen members per broth
BROTH_MIN_MEMBERS = 80
BROTH_MAX_MEMBERS = 180

# Sparsity parameters
# Uptake: fraction of nutrients a species CAN uptake
UPTAKE_SPARSITY = {
    "pathogen"   : (0.30, 0.55),   # 30–55 % of nutrients
    "specialist" : (0.10, 0.25),   # 10–25 % (but preferred nutrient always on)
    "generalist" : (0.40, 0.65),   # 40–65 %
}
# Cross-feeding: how many output slots receive flux when consuming nutrient β
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 8   # out of N_RESOURCES

RESOURCE_NAMES = [f"c{k+1}"      for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads"

TYPE_COLORS = {
    "pathogen"   : "#E53935",
    "specialist" : "#1E88E5",
    "generalist" : "#43A047",
}

In [2]:
# =============================================================================
# 1.  SPECIES TRAITS  (sparse uptake)
# =============================================================================

def build_species_traits():
    """
    Binary genomes → sigmoid uptake rates, then masked to sparse support.

    Sparsity rules
    ──────────────
    • Each species gets a Bernoulli mask whose density is drawn from the
      type-specific range in UPTAKE_SPARSITY.
    • Specialists always have their preferred nutrient active.
    • Nutrients outside the mask are forced to exactly 0.
    """
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix           = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask        = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type       = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype = "pathogen"
            pref  = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"
            pref  = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"
            pref  = -1

        species_type.append(stype)
        preferred_nutrient.append(pref)

        # Draw sparsity density
        lo, hi = UPTAKE_SPARSITY[stype]
        density = rng.uniform(lo, hi)
        n_active = max(1, int(round(density * N_RESOURCES)))

        # Build mask
        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active_indices:
            # Specialists must have their preferred nutrient
            active_indices[0] = pref
        uptake_mask[i, active_indices] = True

        # Compute sigmoid uptake only for active nutrients
        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        raw = genes[i].astype(float) @ W + bias
        c_full = 1.0 / (1.0 + np.exp(-raw))
        c_matrix[i] = c_full * uptake_mask[i]   # zero out inactive nutrients

    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient

In [3]:
# =============================================================================
# 2.  BROTH ENVIRONMENTS
# =============================================================================

def build_broths(species_type):
    """
    30 broths, each with:
      • 1 dominant nutrient (high R0)
      • 2–5 absent nutrients (R0 = 0; cross-feeding only)
      • Partial community: random subset of 80–180 non-pathogen species
    """
    broths = []
    for b in range(N_BROTHS):
        dominant    = int(rng.integers(0, N_RESOURCES))
        non_dominant = [k for k in range(N_RESOURCES) if k != dominant]
        n_absent    = int(rng.integers(2, 6))
        absent_set  = set(rng.choice(non_dominant, size=n_absent, replace=False).tolist())

        R0 = np.zeros(N_RESOURCES)
        for k in range(N_RESOURCES):
            if k == dominant:
                R0[k] = rng.uniform(0.5, 2.0) * rng.uniform(3.0, 6.0)
            elif k in absent_set:
                R0[k] = 0.0
            else:
                R0[k] = rng.uniform(0.5, 2.0)

        delta = float(rng.uniform(0.05, 0.20))
        w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
        l     = rng.uniform(0.1, 0.3, size=N_RESOURCES)

        # Partial community membership (non-pathogen species)
        n_members  = int(rng.integers(BROTH_MIN_MEMBERS, BROTH_MAX_MEMBERS + 1))
        comm_pool  = np.arange(1, N_SPECIES)
        members_np = rng.choice(comm_pool, size=n_members, replace=False)
        members    = [PATHOGEN_IDX] + sorted(members_np.tolist())

        broths.append({
            "name"      : BROTH_NAMES[b],
            "delta"     : delta,
            "R0"        : R0,
            "w"         : w,
            "l"         : l,
            "dominant"  : dominant,
            "absent"    : sorted(absent_set),
            "members"   : members,
            "n_members" : len(members),
        })
    return broths

In [4]:
# =============================================================================
# 3.  SPECIES-SPECIFIC SPARSE CROSS-FEEDING MATRICES
# =============================================================================

def build_species_crossfeed_matrices(species_type):
    """
    One sparse D_i per species.

    Sparsity rule
    ─────────────
    For each consumed nutrient β, only CROSSFEED_OUT_MIN–CROSSFEED_OUT_MAX
    output nutrients α receive non-zero flux.  The active α set is drawn
    without replacement.  The Dirichlet draw is over that subset and the
    probabilities renormalized so each column sums to 1 (mass conservation).
    Off-mask entries stay 0.
    """
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))

        for beta in range(N_RESOURCES):
            # How many output slots are active for this (species, input)?
            n_out = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            # Dirichlet concentration by type
            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.ones(n_out)
                for j, a in enumerate(active_alpha):
                    if a == pref and beta != pref:
                        alpha_vec[j] = rng.uniform(2.0, 5.0)
                    else:
                        alpha_vec[j] = rng.uniform(0.5, 2.0)
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            probs = rng.dirichlet(alpha_vec)          # sums to 1 over active set
            D_i[active_alpha, beta] = probs

        D_list.append(D_i)
    return D_list

In [5]:
# =============================================================================
# 4.  ODE
# =============================================================================

def micrm_ode(t, y, c, w, l, m, g, delta, R0, D_list_sub):
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp],  0.0)
    R = np.maximum(y[n_sp:],  0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    eps = 1e-12
    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        Ni = N[i]
        if Ni < 1e-14:
            continue
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            flux_i_beta = l[beta] * c[i, beta] * Ni * R[beta]
            if flux_i_beta < 1e-16:
                continue
            # sparse: only non-zero rows of D_list_sub[i][:, beta]
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]
            for alpha in nz:
                crossfeed[alpha] += (col[alpha]
                                     * w[beta] / (w[alpha] + eps)
                                     * flux_i_beta)
    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])

In [6]:
# =============================================================================
# 5.  ODE RUNNER
# =============================================================================

def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=500.0):
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]
    n_sub      = len(species_subset)

    y0 = np.concatenate([np.full(n_sub, 0.01), R0.copy()])
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, 400),
        rtol   = 1e-5,
        atol   = 1e-7,
    )
    y_final = sol.y[:, -1]
    N_ss    = np.maximum(y_final[:n_sub], 0.0)
    R_final = np.maximum(y_final[n_sub:], 0.0)

    N_out = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = N_ss[local_idx]
    return N_out, R_final, sol

In [7]:
# =============================================================================
# 6.  METRIC FUNCTIONS
# =============================================================================

def calc_pairwise_suppression(N_pathogen_pair, N_pathogen_solo):
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    return float(1.0 - (N_pathogen_pair / (N_pathogen_solo + eps)))


def calc_community_suppression(N_pathogen_comm, N_pathogen_solo):
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    return float(1.0 - (N_pathogen_comm / (N_pathogen_solo + eps)))

In [8]:
# =============================================================================
# 7.  MAIN DATASET GENERATOR
# =============================================================================

def generate_dataset():
    print("\n[1] Generating species traits (sparse uptake) …")
    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()
    m = rng.uniform(0.2, 0.6, size=N_SPECIES)
    g = rng.uniform(0.8, 1.2, size=N_SPECIES)

    type_counts = {t: species_type.count(t) for t in ["pathogen", "specialist", "generalist"]}
    print(f"    Species counts: {type_counts}")
    avg_active = uptake_mask.mean(axis=1) * N_RESOURCES
    print(f"    Mean active nutrients per species: {avg_active.mean():.1f}  "
          f"(min={avg_active.min():.0f}, max={avg_active.max():.0f})")

    print("\n[2] Building sparse species-specific D_i matrices …")
    D_list = build_species_crossfeed_matrices(species_type)
    # Sparsity report
    nz_fracs = [(D_list[i] > 0).mean() for i in range(N_SPECIES)]
    print(f"    Mean D_i non-zero fraction: {np.mean(nz_fracs):.3f}  "
          f"(vs {1.0:.3f} fully dense)")

    print("\n[3] Generating broth environments (partial membership) …")
    broths = build_broths(species_type)
    for b in broths:
        absent_str = [RESOURCE_NAMES[k] for k in b["absent"]]
        print(f"    {b['name']}: dominant={RESOURCE_NAMES[b['dominant']]}, "
              f"absent={absent_str}, n_members={b['n_members']}, delta={b['delta']:.3f}")

    # ── Pathogen solo ─────────────────────────────────────────────────────
    print("\n[4] Computing pathogen solo abundances …")
    pathogen_solo = {}
    for broth in tqdm(broths, desc="  Pathogen solo", unit="broth"):
        N_s, _, _ = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=[PATHOGEN_IDX], t_end=300.0,
        )
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])
        print(f"    {broth['name']}: N_pathogen_solo = {N_s[PATHOGEN_IDX]:.4f}")

    # ── Pairwise ODE runs ─────────────────────────────────────────────────
    print("\n[5] Running pairwise ODE (species_i + pathogen) …")
    print("    (only species present in each broth are tested)")
    pairwise_rows   = []
    pairwise_matrix = {}

    total_pairs = sum(len([sp for sp in b["members"] if sp != PATHOGEN_IDX]) for b in broths)
    with tqdm(total=total_pairs, desc="  Pairwise ODE", unit="run") as pbar:
        for b_idx, broth in enumerate(broths):
            bname        = broth["name"]
            N_path_solo  = pathogen_solo[bname]
            pairwise_matrix[bname] = {}

            broth_species = [sp for sp in broth["members"] if sp != PATHOGEN_IDX]

            for sp_idx in broth_species:
                pbar.set_postfix(broth=broth["name"].replace("Broth_","B"),
                                 sp=SPECIES_NAMES[sp_idx].replace("Species_","S"))
                N_pair, _, _ = run_micrm(
                    c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
                    delta=broth["delta"], R0=broth["R0"], D_list=D_list,
                    species_subset=[sp_idx, PATHOGEN_IDX], t_end=300.0,
                )
                N_pathogen_pair = float(N_pair[PATHOGEN_IDX])
                pw_score = calc_pairwise_suppression(N_pathogen_pair, N_path_solo)
                pairwise_matrix[bname][sp_idx] = pw_score

                row = {
                    "species_id"        : SPECIES_NAMES[sp_idx],
                    "broth_id"          : bname,
                    "species_type"      : species_type[sp_idx],
                    "dominant_nutrient" : RESOURCE_NAMES[broth["dominant"]],
                    "absent_nutrients"  : str([RESOURCE_NAMES[k] for k in broth["absent"]]),
                    "delta_per_h"       : round(broth["delta"], 4),
                    "maintenance_cost"  : round(float(m[sp_idx]), 4),
                }
                for g_idx in range(N_GENES):
                    row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
                for k in range(N_RESOURCES):
                    row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[sp_idx, k]), 4)
                for k in range(N_RESOURCES):
                    row[f"R0_{RESOURCE_NAMES[k]}"] = round(float(broth["R0"][k]), 4)

                row["suppression_score"] = round(pw_score, 4)
                pairwise_rows.append(row)
                pbar.update(1)

            scores = list(pairwise_matrix[bname].values())
            if scores:
                tqdm.write(f"    {bname} ({len(broth_species)} species): "
                           f"scores [{min(scores):.3f}, {max(scores):.3f}]  "
                           f"path_solo={N_path_solo:.4f}")

    df = pd.DataFrame(pairwise_rows)

    # ── Full community ODE ────────────────────────────────────────────────
    print("\n[6] Running full community ODE (broth-specific member sets) …")
    community_N   = {}
    comm_supp_map = {}

    for broth in tqdm(broths, desc="  Community ODE", unit="broth"):
        bname = broth["name"]
        N_comm, R_ss, sol_comm = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=broth["members"], t_end=400.0,
        )
        community_N[bname]   = (N_comm, R_ss, sol_comm)
        comm_supp             = calc_community_suppression(
            N_comm[PATHOGEN_IDX], pathogen_solo[bname])
        comm_supp_map[bname] = comm_supp
        tqdm.write(f"    {bname}: community suppression = {comm_supp:.3f}  "
                   f"N_pathogen_comm = {N_comm[PATHOGEN_IDX]:.4f}  "
                   f"n_members={broth['n_members']}")

    df["community_pathogen_suppression"] = df["broth_id"].map(comm_supp_map)

    # ── Final column order ────────────────────────────────────────────────
    gene_cols   = [f"gene_{g+1}"  for g in range(N_GENES)]
    uptake_cols = [f"uptake_{r}"  for r in RESOURCE_NAMES]
    R0_cols     = [f"R0_{r}"      for r in RESOURCE_NAMES]
    col_order = (
        ["species_id", "broth_id",
         "species_type", "dominant_nutrient", "absent_nutrients",
         "delta_per_h", "maintenance_cost"]
        + gene_cols + uptake_cols + R0_cols
        + ["suppression_score", "community_pathogen_suppression"]
    )
    df = df[col_order]

    print(f"\n  Dataset shape : {df.shape}")
    print(f"  suppression_score summary:\n{df['suppression_score'].describe().round(3)}")
    print(f"\n  Fraction suppressing pathogen (score > 0): "
          f"{(df['suppression_score'] > 0).mean():.2%}")

    return (df, genes, c_matrix, uptake_mask, m, g, broths,
            D_list, species_type, preferred_nutrient,
            pathogen_solo, pairwise_matrix, community_N, comm_supp_map)

In [10]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

# ── Global constants ──────────────────────────────────────────────────────────
N_SPECIES    = 1000          # 1 pathogen + community
N_RESOURCES  = 30
N_GENES      = 8
N_BROTHS     = 30

PATHOGEN_IDX  = 0
N_COMMUNITY   = N_SPECIES - 1
N_SPECIALIST  = N_COMMUNITY // 3          # ~333 specialists
N_GENERALIST  = N_COMMUNITY - N_SPECIALIST  # ~666 generalists

# Partial community: how many non-pathogen members per broth
BROTH_MIN_MEMBERS = 80
BROTH_MAX_MEMBERS = 180

# Sparsity parameters
# Uptake: fraction of nutrients a species CAN uptake
UPTAKE_SPARSITY = {
    "pathogen"   : (0.30, 0.55),   # 30–55 % of nutrients
    "specialist" : (0.10, 0.25),   # 10–25 % (but preferred nutrient always on)
    "generalist" : (0.40, 0.65),   # 40–65 %
}
# Cross-feeding: how many output slots receive flux when consuming nutrient β
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 8   # out of N_RESOURCES

RESOURCE_NAMES = [f"c{k+1}"      for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads"

TYPE_COLORS = {
    "pathogen"   : "#E53935",
    "specialist" : "#1E88E5",
    "generalist" : "#43A047",
}


# =============================================================================
# 1.  SPECIES TRAITS  (sparse uptake)
# =============================================================================

def build_species_traits():
    """
    Binary genomes → sigmoid uptake rates, then masked to sparse support.

    Sparsity rules
    ──────────────
    • Each species gets a Bernoulli mask whose density is drawn from the
      type-specific range in UPTAKE_SPARSITY.
    • Specialists always have their preferred nutrient active.
    • Nutrients outside the mask are forced to exactly 0.
    """
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix           = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask        = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type       = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype = "pathogen"
            pref  = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"
            pref  = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"
            pref  = -1

        species_type.append(stype)
        preferred_nutrient.append(pref)

        # Draw sparsity density
        lo, hi = UPTAKE_SPARSITY[stype]
        density = rng.uniform(lo, hi)
        n_active = max(1, int(round(density * N_RESOURCES)))

        # Build mask
        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active_indices:
            # Specialists must have their preferred nutrient
            active_indices[0] = pref
        uptake_mask[i, active_indices] = True

        # Compute sigmoid uptake only for active nutrients
        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        raw = genes[i].astype(float) @ W + bias
        c_full = 1.0 / (1.0 + np.exp(-raw))
        c_matrix[i] = c_full * uptake_mask[i]   # zero out inactive nutrients

    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient


# =============================================================================
# 2.  BROTH ENVIRONMENTS
# =============================================================================

def build_broths(species_type):
    """
    30 broths, each with:
      • 1 dominant nutrient (high R0)
      • 2–5 absent nutrients (R0 = 0; cross-feeding only)
      • Partial community: random subset of 80–180 non-pathogen species
    """
    broths = []
    for b in range(N_BROTHS):
        dominant    = int(rng.integers(0, N_RESOURCES))
        non_dominant = [k for k in range(N_RESOURCES) if k != dominant]
        n_absent    = int(rng.integers(2, 6))
        absent_set  = set(rng.choice(non_dominant, size=n_absent, replace=False).tolist())

        R0 = np.zeros(N_RESOURCES)
        for k in range(N_RESOURCES):
            if k == dominant:
                R0[k] = rng.uniform(0.5, 2.0) * rng.uniform(3.0, 6.0)
            elif k in absent_set:
                R0[k] = 0.0
            else:
                R0[k] = rng.uniform(0.5, 2.0)

        delta = float(rng.uniform(0.05, 0.20))
        w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
        l     = rng.uniform(0.1, 0.3, size=N_RESOURCES)

        # Partial community membership (non-pathogen species)
        n_members  = int(rng.integers(BROTH_MIN_MEMBERS, BROTH_MAX_MEMBERS + 1))
        comm_pool  = np.arange(1, N_SPECIES)
        members_np = rng.choice(comm_pool, size=n_members, replace=False)
        members    = [PATHOGEN_IDX] + sorted(members_np.tolist())

        broths.append({
            "name"      : BROTH_NAMES[b],
            "delta"     : delta,
            "R0"        : R0,
            "w"         : w,
            "l"         : l,
            "dominant"  : dominant,
            "absent"    : sorted(absent_set),
            "members"   : members,
            "n_members" : len(members),
        })
    return broths


# =============================================================================
# 3.  SPECIES-SPECIFIC SPARSE CROSS-FEEDING MATRICES
# =============================================================================

def build_species_crossfeed_matrices(species_type):
    """
    One sparse D_i per species.

    Sparsity rule
    ─────────────
    For each consumed nutrient β, only CROSSFEED_OUT_MIN–CROSSFEED_OUT_MAX
    output nutrients α receive non-zero flux.  The active α set is drawn
    without replacement.  The Dirichlet draw is over that subset and the
    probabilities renormalized so each column sums to 1 (mass conservation).
    Off-mask entries stay 0.
    """
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))

        for beta in range(N_RESOURCES):
            # How many output slots are active for this (species, input)?
            n_out = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            # Dirichlet concentration by type
            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.ones(n_out)
                for j, a in enumerate(active_alpha):
                    if a == pref and beta != pref:
                        alpha_vec[j] = rng.uniform(2.0, 5.0)
                    else:
                        alpha_vec[j] = rng.uniform(0.5, 2.0)
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            probs = rng.dirichlet(alpha_vec)          # sums to 1 over active set
            D_i[active_alpha, beta] = probs

        D_list.append(D_i)
    return D_list


# =============================================================================
# 4.  ODE
# =============================================================================

def micrm_ode(t, y, c, w, l, m, g, delta, R0, D_list_sub):
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp],  0.0)
    R = np.maximum(y[n_sp:],  0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    eps = 1e-12
    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        Ni = N[i]
        if Ni < 1e-14:
            continue
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            flux_i_beta = l[beta] * c[i, beta] * Ni * R[beta]
            if flux_i_beta < 1e-16:
                continue
            # sparse: only non-zero rows of D_list_sub[i][:, beta]
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]
            for alpha in nz:
                crossfeed[alpha] += (col[alpha]
                                     * w[beta] / (w[alpha] + eps)
                                     * flux_i_beta)
    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


# =============================================================================
# 5.  ODE RUNNER
# =============================================================================

def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=500.0):
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]
    n_sub      = len(species_subset)

    y0 = np.concatenate([np.full(n_sub, 0.01), R0.copy()])
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, 400),
        rtol   = 1e-5,
        atol   = 1e-7,
    )
    y_final = sol.y[:, -1]
    N_ss    = np.maximum(y_final[:n_sub], 0.0)
    R_final = np.maximum(y_final[n_sub:], 0.0)

    N_out = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = N_ss[local_idx]
    return N_out, R_final, sol


# =============================================================================
# 6.  METRIC FUNCTIONS
# =============================================================================

def calc_pairwise_suppression(N_pathogen_pair, N_pathogen_solo):
    """
    Fraction by which a companion species reduces pathogen abundance vs solo.
    Clipped to [0, 1]: species that facilitate the pathogen score 0, not negative.
    """
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    raw = 1.0 - (N_pathogen_pair / (N_pathogen_solo + eps))
    return float(max(0.0, raw))


def calc_community_suppression(N_pathogen_comm, N_pathogen_solo):
    eps = 1e-10
    if N_pathogen_solo < eps:
        return 0.0
    raw = 1.0 - (N_pathogen_comm / (N_pathogen_solo + eps))
    return float(max(0.0, raw))


# =============================================================================
# 7.  MAIN DATASET GENERATOR
# =============================================================================

def generate_dataset():
    print("\n[1] Generating species traits (sparse uptake) …")
    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()
    m = rng.uniform(0.2, 0.6, size=N_SPECIES)
    g = rng.uniform(0.8, 1.2, size=N_SPECIES)

    type_counts = {t: species_type.count(t) for t in ["pathogen", "specialist", "generalist"]}
    print(f"    Species counts: {type_counts}")
    avg_active = uptake_mask.mean(axis=1) * N_RESOURCES
    print(f"    Mean active nutrients per species: {avg_active.mean():.1f}  "
          f"(min={avg_active.min():.0f}, max={avg_active.max():.0f})")

    print("\n[2] Building sparse species-specific D_i matrices …")
    D_list = build_species_crossfeed_matrices(species_type)
    # Sparsity report
    nz_fracs = [(D_list[i] > 0).mean() for i in range(N_SPECIES)]
    print(f"    Mean D_i non-zero fraction: {np.mean(nz_fracs):.3f}  "
          f"(vs {1.0:.3f} fully dense)")

    print("\n[3] Generating broth environments (partial membership) …")
    broths = build_broths(species_type)
    for b in broths:
        absent_str = [RESOURCE_NAMES[k] for k in b["absent"]]
        print(f"    {b['name']}: dominant={RESOURCE_NAMES[b['dominant']]}, "
              f"absent={absent_str}, n_members={b['n_members']}, delta={b['delta']:.3f}")

    # ── Pathogen solo ─────────────────────────────────────────────────────
    print("\n[4] Computing pathogen solo abundances …")
    pathogen_solo = {}
    for broth in tqdm(broths, desc="  Pathogen solo", unit="broth"):
        N_s, _, _ = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=[PATHOGEN_IDX], t_end=300.0,
        )
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])
        print(f"    {broth['name']}: N_pathogen_solo = {N_s[PATHOGEN_IDX]:.4f}")

    # ── Pairwise ODE runs ─────────────────────────────────────────────────
    print("\n[5] Running pairwise ODE (species_i + pathogen) …")
    print("    (only species present in each broth are tested)")
    pairwise_rows   = []
    pairwise_matrix = {}

    total_pairs = sum(len([sp for sp in b["members"] if sp != PATHOGEN_IDX]) for b in broths)
    with tqdm(total=total_pairs, desc="  Pairwise ODE", unit="run") as pbar:
        for b_idx, broth in enumerate(broths):
            bname        = broth["name"]
            N_path_solo  = pathogen_solo[bname]
            pairwise_matrix[bname] = {}

            broth_species = [sp for sp in broth["members"] if sp != PATHOGEN_IDX]

            for sp_idx in broth_species:
                pbar.set_postfix(broth=broth["name"].replace("Broth_","B"),
                                 sp=SPECIES_NAMES[sp_idx].replace("Species_","S"))
                N_pair, _, _ = run_micrm(
                    c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
                    delta=broth["delta"], R0=broth["R0"], D_list=D_list,
                    species_subset=[sp_idx, PATHOGEN_IDX], t_end=300.0,
                )
                N_pathogen_pair = float(N_pair[PATHOGEN_IDX])
                pw_score = calc_pairwise_suppression(N_pathogen_pair, N_path_solo)
                pairwise_matrix[bname][sp_idx] = pw_score

                row = {
                    "species_id"        : SPECIES_NAMES[sp_idx],
                    "broth_id"          : bname,
                    "species_type"      : species_type[sp_idx],
                    "dominant_nutrient" : RESOURCE_NAMES[broth["dominant"]],
                    "absent_nutrients"  : str([RESOURCE_NAMES[k] for k in broth["absent"]]),
                    "delta_per_h"       : round(broth["delta"], 4),
                    "maintenance_cost"  : round(float(m[sp_idx]), 4),
                }
                for g_idx in range(N_GENES):
                    row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
                for k in range(N_RESOURCES):
                    row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[sp_idx, k]), 4)
                for k in range(N_RESOURCES):
                    row[f"R0_{RESOURCE_NAMES[k]}"] = round(float(broth["R0"][k]), 4)

                row["suppression_score"] = round(pw_score, 4)
                pairwise_rows.append(row)
                pbar.update(1)

            scores = list(pairwise_matrix[bname].values())
            if scores:
                tqdm.write(f"    {bname} ({len(broth_species)} species): "
                           f"scores [{min(scores):.3f}, {max(scores):.3f}]  "
                           f"path_solo={N_path_solo:.4f}")

    df = pd.DataFrame(pairwise_rows)

    # ── Full community ODE ────────────────────────────────────────────────
    print("\n[6] Running full community ODE (broth-specific member sets) …")
    community_N   = {}
    comm_supp_map = {}

    for broth in tqdm(broths, desc="  Community ODE", unit="broth"):
        bname = broth["name"]
        N_comm, R_ss, sol_comm = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=broth["members"], t_end=400.0,
        )
        community_N[bname]   = (N_comm, R_ss, sol_comm)
        comm_supp             = calc_community_suppression(
            N_comm[PATHOGEN_IDX], pathogen_solo[bname])
        comm_supp_map[bname] = comm_supp
        tqdm.write(f"    {bname}: community suppression = {comm_supp:.3f}  "
                   f"N_pathogen_comm = {N_comm[PATHOGEN_IDX]:.4f}  "
                   f"n_members={broth['n_members']}")

    df["community_pathogen_suppression"] = df["broth_id"].map(comm_supp_map)

    # ── Final column order ────────────────────────────────────────────────
    gene_cols   = [f"gene_{g+1}"  for g in range(N_GENES)]
    uptake_cols = [f"uptake_{r}"  for r in RESOURCE_NAMES]
    R0_cols     = [f"R0_{r}"      for r in RESOURCE_NAMES]
    col_order = (
        ["species_id", "broth_id",
         "species_type", "dominant_nutrient", "absent_nutrients",
         "delta_per_h", "maintenance_cost"]
        + gene_cols + uptake_cols + R0_cols
        + ["suppression_score", "community_pathogen_suppression"]
    )
    df = df[col_order]

    print(f"\n  Dataset shape : {df.shape}")
    print(f"  suppression_score summary:\n{df['suppression_score'].describe().round(3)}")
    print(f"\n  Fraction suppressing pathogen (score > 0): "
          f"{(df['suppression_score'] > 0).mean():.2%}")

    return (df, genes, c_matrix, uptake_mask, m, g, broths,
            D_list, species_type, preferred_nutrient,
            pathogen_solo, pairwise_matrix, community_N, comm_supp_map)


# =============================================================================
# 8.  FIGURES  (adapted for larger scale)
# =============================================================================

def make_figures(df, genes, c_matrix, uptake_mask, m, g, broths,
                 D_list, species_type, preferred_nutrient,
                 pathogen_solo, pairwise_matrix, community_N, comm_supp_map):

    print("\n[7] Generating figures …")

    # ── FIG 1: Full community ODE time series ─────────────────────────────
    b0     = broths[0]
    bname0 = b0["name"]
    _, _, sol = community_N[bname0]
    n_members0 = b0["n_members"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for local_i, global_i in enumerate(b0["members"]):
        col = TYPE_COLORS[species_type[global_i]]
        lw  = 2.5 if global_i == PATHOGEN_IDX else 0.5
        zo  = 5   if global_i == PATHOGEN_IDX else 1
        al  = 1.0 if global_i == PATHOGEN_IDX else 0.25
        axes[0].plot(sol.t, sol.y[local_i], color=col, lw=lw, alpha=al, zorder=zo)
    from matplotlib.lines import Line2D
    axes[0].legend(handles=[
        Line2D([0],[0], color=TYPE_COLORS["pathogen"],   lw=2.5, label="Pathogen"),
        Line2D([0],[0], color=TYPE_COLORS["specialist"], lw=1.5, label="Specialist"),
        Line2D([0],[0], color=TYPE_COLORS["generalist"], lw=1.5, label="Generalist"),
    ], fontsize=8)
    axes[0].set_xlabel("Time (h)")
    axes[0].set_ylabel("Abundance N_i")
    absent_str = [RESOURCE_NAMES[k] for k in b0["absent"]]
    axes[0].set_title(
        f"Community Dynamics — {bname0} (n={n_members0} members)\n"
        f"dominant={RESOURCE_NAMES[b0['dominant']]}, absent={absent_str}",
        fontweight="bold")
    axes[0].set_yscale("symlog", linthresh=0.001)
    axes[0].grid(True, alpha=0.3)

    res_colors = plt.cm.tab20(np.linspace(0, 1, N_RESOURCES))
    for k in range(N_RESOURCES):
        lbl = f"{RESOURCE_NAMES[k]}{' (absent)' if k in b0['absent'] else ''}"
        axes[1].plot(sol.t, sol.y[n_members0 + k],
                     color=res_colors[k], lw=1.8, label=lbl)
    axes[1].set_xlabel("Time (h)")
    axes[1].set_ylabel("Nutrient concentration R_α")
    axes[1].set_title("Nutrient Dynamics (30 nutrients)\n"
                      "(absent nutrients supplied only by cross-feeding)", fontweight="bold")
    axes[1].legend(fontsize=5, ncol=2)
    axes[1].grid(True, alpha=0.3)
    fig.suptitle("Figure 1: Full Community MiCRM Dynamics", fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig1_ode_dynamics_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig1_ode_dynamics_v5.png")

    # ── FIG 2: Uptake sparsity overview ───────────────────────────────────
    # Show a 100-species × 30-nutrient subset
    sample_idx = [PATHOGEN_IDX] + list(range(1, 60)) + list(range(N_SPECIALIST+1, N_SPECIALIST+41))
    sample_idx = sample_idx[:80]
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig.suptitle("Figure 2: Sparse Uptake Rate Heatmap (80-species sample)",
                 fontsize=12, fontweight="bold")
    row_colors = [TYPE_COLORS[species_type[i]] for i in sample_idx]
    sns.heatmap(c_matrix[sample_idx, :], ax=axes[0], cmap="YlOrRd", vmin=0, vmax=1,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[SPECIES_NAMES[i].replace("Species_","S") for i in sample_idx],
                linewidths=0.2, linecolor="white",
                cbar_kws={"label": "Uptake rate c_iα (0 = masked out)"})
    axes[0].set_title("Uptake Rates c_iα\n(white = nutrient not available to species)",
                      fontweight="bold")
    for tick, color in zip(axes[0].get_yticklabels(), row_colors):
        tick.set_color(color)
    axes[0].tick_params(axis="y", labelsize=6)

    # Active nutrient count distribution
    active_counts = uptake_mask.sum(axis=1)
    for stype, col in TYPE_COLORS.items():
        idx = [i for i, t in enumerate(species_type) if t == stype]
        axes[1].hist([active_counts[i] for i in idx], bins=20,
                     color=col, alpha=0.6, label=stype.capitalize(), edgecolor="none")
    axes[1].set_xlabel("Number of active nutrients per species")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Uptake Sparsity Distribution\nby Species Type", fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig2_uptake_sparsity_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig2_uptake_sparsity_v5.png")

    # ── FIG 3: Sparse D_i matrices for 8 representative species ──────────
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Figure 3: Sparse Cross-Feeding Matrices D_i (selected species)\n"
                 "(white = no flux; each col sums to ≤1)",
                 fontsize=12, fontweight="bold")
    sample_sp = [0, 1, 5, 10, N_SPECIALIST+1, N_SPECIALIST+5, N_SPECIALIST+20, N_SPECIALIST+50]
    for ax_idx, sp_idx in enumerate(sample_sp):
        sns.heatmap(D_list[sp_idx], ax=axes.flatten()[ax_idx],
                    cmap="YlOrRd", vmin=0, vmax=0.8,
                    xticklabels=[f"in {r}" for r in RESOURCE_NAMES],
                    yticklabels=[f"out {r}" for r in RESOURCE_NAMES],
                    linewidths=0.3, cbar=(ax_idx == 7))
        axes.flatten()[ax_idx].set_title(
            f"{SPECIES_NAMES[sp_idx]} [{species_type[sp_idx]}]",
            fontweight="bold", color=TYPE_COLORS[species_type[sp_idx]], fontsize=9)
        axes.flatten()[ax_idx].tick_params(axis="both", labelsize=5)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig3_sparse_crossfeed_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig3_sparse_crossfeed_v5.png")

    # ── FIG 4: Community membership overview ──────────────────────────────
    membership = np.zeros((N_BROTHS, N_SPECIES), dtype=np.int8)
    for b_idx, broth in enumerate(broths):
        for sp in broth["members"]:
            membership[b_idx, sp] = 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Figure 4: Partial Community Membership",
                 fontsize=12, fontweight="bold")

    # Heatmap for first 100 species
    sns.heatmap(membership[:, :100], ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=[SPECIES_NAMES[i].replace("Species_","S") for i in range(100)],
                yticklabels=[b["name"].replace("Broth_","B") for b in broths],
                linewidths=0.1, cbar=False)
    axes[0].set_title("Membership heatmap (first 100 species)\nBlue = present in broth",
                      fontweight="bold")
    axes[0].tick_params(axis="x", labelsize=5, rotation=90)

    n_members_list = [b["n_members"] for b in broths]
    axes[1].bar(range(N_BROTHS), n_members_list, color="#7B1FA2", alpha=0.8, edgecolor="white")
    axes[1].axhline(np.mean(n_members_list), color="k", ls="--", lw=1.5,
                    label=f"Mean = {np.mean(n_members_list):.0f}")
    axes[1].set_xticks(range(N_BROTHS))
    axes[1].set_xticklabels([b["name"].replace("Broth_","B") for b in broths],
                             rotation=60, fontsize=7)
    axes[1].set_ylabel("Number of species (incl. pathogen)")
    axes[1].set_title("Community Size per Broth", fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig4_membership_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig4_membership_v5.png")

    # ── FIG 5: Pairwise suppression heatmap (species × broth) ─────────────
    # Build pivot: only (species, broth) pairs that were actually tested
    pivot_pw = df.pivot_table(index="species_id", columns="broth_id",
                              values="suppression_score", aggfunc="mean")
    # Sample 60 species for readability
    sample_sp_ids = (df.groupby("species_id")["suppression_score"].mean()
                       .sort_values().index.tolist())
    # Take 20 from each end + 20 middle
    n = len(sample_sp_ids)
    show_ids = (sample_sp_ids[:20]
                + sample_sp_ids[n//2-10:n//2+10]
                + sample_sp_ids[-20:])
    show_ids = list(dict.fromkeys(show_ids))[:60]
    pivot_show = pivot_pw.loc[[s for s in show_ids if s in pivot_pw.index]]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle("Figure 5: Pairwise Suppression Heatmap (60-species sample)",
                 fontsize=12, fontweight="bold")
    sns.heatmap(pivot_show, ax=axes[0], cmap="RdYlGn", center=0,
                vmin=-0.5, vmax=1.0, linewidths=0.1, linecolor="lightgray",
                cbar_kws={"label": "Pairwise suppression score"},
                annot=False)
    axes[0].set_title("Green = suppresses pathogen | Red = facilitates\n"
                      "White = not tested in this broth", fontweight="bold")
    axes[0].tick_params(axis="y", labelsize=6)

    mean_pw_df = (df.groupby(["species_id", "species_type"])["suppression_score"]
                    .mean().reset_index()
                    .sort_values("suppression_score", ascending=True))
    # Show top 40 suppressors and bottom 40
    show_df = pd.concat([mean_pw_df.head(30), mean_pw_df.tail(30)])
    axes[1].barh(range(len(show_df)), show_df["suppression_score"],
                 color=[TYPE_COLORS[t] for t in show_df["species_type"]],
                 alpha=0.85, edgecolor="white")
    axes[1].set_yticks(range(len(show_df)))
    axes[1].set_yticklabels(
        [s.replace("Species_", "S") for s in show_df["species_id"]], fontsize=7)
    axes[1].axvline(0, color="black", lw=1.5, ls="--")
    axes[1].set_xlabel("Mean pairwise suppression score")
    axes[1].set_title("Ranked Mean Pairwise Suppression\n(top 30 suppressors + bottom 30)",
                      fontweight="bold")
    axes[1].grid(True, alpha=0.3, axis="x")
    from matplotlib.patches import Patch
    axes[1].legend(handles=[
        Patch(color=TYPE_COLORS["specialist"],  label="Specialist"),
        Patch(color=TYPE_COLORS["generalist"],  label="Generalist"),
    ], fontsize=8)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig5_pairwise_suppression_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig5_pairwise_suppression_v5.png")

    # ── FIG 6: Ecological summary ──────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Figure 6: Ecological Summary — Suppression & Traits",
                 fontsize=13, fontweight="bold")

    # 6a community suppression per broth
    broth_names = [b["name"] for b in broths]
    comm_supps  = [comm_supp_map[bn] for bn in broth_names]
    axes[0,0].bar(range(N_BROTHS), comm_supps, color="#7B1FA2", alpha=0.8, edgecolor="white")
    for xi, (v, b) in enumerate(zip(comm_supps, broths)):
        absent_str = ",".join([RESOURCE_NAMES[k] for k in b["absent"]])
        axes[0,0].text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=6, rotation=90)
    axes[0,0].set_xticks(range(N_BROTHS))
    axes[0,0].set_xticklabels([bn.replace("Broth_","B") for bn in broth_names],
                               rotation=60, fontsize=7)
    axes[0,0].set_ylabel("Community pathogen suppression")
    axes[0,0].set_title("Community Pathogen Suppression per Broth", fontweight="bold")
    axes[0,0].set_ylim(-0.15, 1.15)
    axes[0,0].grid(True, alpha=0.3, axis="y")

    # 6b suppression distribution by type
    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        sub = df[df["species_type"] == stype]["suppression_score"]
        axes[0,1].hist(sub, bins=30, color=col, alpha=0.55,
                       label=f"{stype.capitalize()} (n={len(sub)})", edgecolor="none")
    axes[0,1].axvline(0, color="black", lw=1.5, ls="--", label="Neutral")
    axes[0,1].set_xlabel("Pairwise suppression score")
    axes[0,1].set_ylabel("Count")
    axes[0,1].set_title("Pairwise Suppression Distribution\nby Species Type", fontweight="bold")
    axes[0,1].legend(fontsize=8)
    axes[0,1].grid(True, alpha=0.3)

    # 6c maintenance cost vs mean pairwise suppression
    mean_pw   = df.groupby("species_id")["suppression_score"].mean()
    sp_ids    = list(mean_pw.index)
    sp_global = [SPECIES_NAMES.index(s) for s in sp_ids]
    m_vals    = [m[i] for i in sp_global]
    sc_col    = [TYPE_COLORS[species_type[i]] for i in sp_global]
    axes[1,0].scatter(m_vals, mean_pw.values, c=sc_col, s=15, alpha=0.5, edgecolors="none")
    axes[1,0].set_xlabel("Maintenance cost m_i")
    axes[1,0].set_ylabel("Mean pairwise suppression")
    axes[1,0].set_title("Maintenance Cost vs Mean Suppression", fontweight="bold")
    axes[1,0].grid(True, alpha=0.3)

    # 6d D_i sparsity (non-zero fraction) vs mean suppression
    nz_fracs_comm = [(D_list[i] > 0).mean() for i in sp_global]
    axes[1,1].scatter(nz_fracs_comm, mean_pw.values,
                      c=sc_col, s=15, alpha=0.5, edgecolors="none")
    axes[1,1].set_xlabel("D_i non-zero fraction\n(cross-feeding output breadth)")
    axes[1,1].set_ylabel("Mean pairwise suppression")
    axes[1,1].set_title("D_i Breadth vs Suppression", fontweight="bold")
    axes[1,1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig6_ecological_summary_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig6_ecological_summary_v5.png")

    # ── FIG 7: Pairwise vs community suppression ──────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Figure 7: Pairwise vs Community Suppression",
                 fontsize=12, fontweight="bold")
    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        sub = df[df["species_type"] == stype]
        axes[0].scatter(sub["suppression_score"],
                        sub["community_pathogen_suppression"],
                        c=col, s=8, alpha=0.3, edgecolors="none",
                        label=stype.capitalize())
    if len(df) > 1:
        r, p = pearsonr(df["suppression_score"], df["community_pathogen_suppression"])
        axes[0].set_title(f"Pairwise vs Community (r={r:.2f}, p={p:.3f})", fontweight="bold")
    axes[0].set_xlabel("Pairwise suppression score")
    axes[0].set_ylabel("Community pathogen suppression")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    broth_mean_pw   = df.groupby("broth_id")["suppression_score"].mean()
    broth_comm_supp = pd.Series(comm_supp_map)
    common_broths   = broth_mean_pw.index.intersection(broth_comm_supp.index)
    axes[1].scatter(broth_mean_pw[common_broths].values,
                    broth_comm_supp[common_broths].values,
                    s=80, c="#7B1FA2", alpha=0.85, edgecolors="k", zorder=5)
    for bn in common_broths:
        axes[1].annotate(bn.replace("Broth_","B"),
                         xy=(broth_mean_pw[bn], broth_comm_supp[bn]),
                         xytext=(3, 3), textcoords="offset points", fontsize=7)
    axes[1].set_xlabel("Mean pairwise suppression (tested species in broth)")
    axes[1].set_ylabel("Community pathogen suppression")
    axes[1].set_title("Broth-level: Mean Pairwise vs Community", fontweight="bold")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig7_pairwise_vs_community_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig7_pairwise_vs_community_v5.png")

    # ── FIG 8: Solo vs Community abundance — does cross-feeding help? ─────
    print("    (running solo ODE for Fig 8 — this may take a few minutes …)")
    solo_N_broth = {}   # {sp_idx: [abundance in each broth it was tested in]}
    comm_N_broth = {}   # {sp_idx: [community abundance in each broth it appears in]}

    # Collect community abundances first (already computed)
    for broth in broths:
        bname = broth["name"]
        N_comm = community_N[bname][0]
        for sp_idx in broth["members"]:
            if sp_idx == PATHOGEN_IDX:
                continue
            comm_N_broth.setdefault(sp_idx, []).append(float(N_comm[sp_idx]))

    # Run solo ODE for each (species, broth) pair that appears in community
    all_pairs = [(sp_idx, broth)
                 for broth in broths
                 for sp_idx in broth["members"]
                 if sp_idx != PATHOGEN_IDX]

    for sp_idx, broth in tqdm(all_pairs, desc="  Solo ODE (Fig 8)", unit="run"):
        N_s, _, _ = run_micrm(
            c_full=c_matrix, w=broth["w"], l=broth["l"], m_full=m, g_full=g,
            delta=broth["delta"], R0=broth["R0"], D_list=D_list,
            species_subset=[sp_idx], t_end=300.0,
        )
        solo_N_broth.setdefault(sp_idx, []).append(float(N_s[sp_idx]))

    # Average across broths for each species
    sp_indices  = sorted(solo_N_broth.keys())
    solo_means  = np.array([np.mean(solo_N_broth[i])  for i in sp_indices])
    comm_means  = np.array([np.mean(comm_N_broth[i])  for i in sp_indices])
    ratio       = np.where(solo_means > 1e-10,
                           comm_means / (solo_means + 1e-10), np.nan)
    sp_types    = [species_type[i] for i in sp_indices]
    bar_colors  = [TYPE_COLORS[t]  for i, t in zip(sp_indices, sp_types)]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(
        "Figure 8: Solo vs Full-Community Steady-State Abundances\n"
        "(Does cross-feeding in a sparse-uptake community increase species abundances?)",
        fontsize=12, fontweight="bold")

    # 8a — scatter solo vs community coloured by type
    for stype, col in [("specialist", TYPE_COLORS["specialist"]),
                       ("generalist",  TYPE_COLORS["generalist"])]:
        idx = [j for j, t in enumerate(sp_types) if t == stype]
        axes[0].scatter(solo_means[idx], comm_means[idx],
                        c=col, s=12, alpha=0.45, edgecolors="none",
                        label=stype.capitalize(), zorder=3)
    lim = max(solo_means.max(), comm_means.max()) * 1.05
    axes[0].plot([0, lim], [0, lim], "k--", lw=1.5, label="No change (1:1)")
    axes[0].set_xlabel("Mean solo abundance  N_solo")
    axes[0].set_ylabel("Mean community abundance  N_comm")
    axes[0].set_title("Solo vs Community\nAbove diagonal = community benefit",
                      fontweight="bold")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # 8b — histogram of log2(N_comm / N_solo) ratio
    log2_ratio = np.log2(ratio[np.isfinite(ratio) & (ratio > 0)])
    for stype, col in [("specialist", TYPE_COLORS["specialist"]),
                       ("generalist",  TYPE_COLORS["generalist"])]:
        idx  = [j for j, t in enumerate(sp_types)
                if t == stype and np.isfinite(ratio[j]) and ratio[j] > 0]
        vals = np.log2(ratio[idx])
        axes[1].hist(vals, bins=40, color=col, alpha=0.55,
                     label=f"{stype.capitalize()} (n={len(vals)})", edgecolor="none")
    axes[1].axvline(0, color="black", lw=1.5, ls="--", label="No change")
    axes[1].set_xlabel("log₂(N_comm / N_solo)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(
        "Community Benefit Distribution\n"
        f"Median log₂ ratio = {np.median(log2_ratio):.2f}  "
        f"({'benefit' if np.median(log2_ratio)>0 else 'cost'})",
        fontweight="bold")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # 8c — mean community abundance vs mean D_i non-zero fraction
    #       to directly test whether broader cross-feeding → higher abundance
    nz_fracs_sp = np.array([(D_list[i] > 0).mean() for i in sp_indices])
    for stype, col in [("specialist", TYPE_COLORS["specialist"]),
                       ("generalist",  TYPE_COLORS["generalist"])]:
        idx = [j for j, t in enumerate(sp_types) if t == stype]
        axes[2].scatter(nz_fracs_sp[idx], comm_means[idx],
                        c=col, s=12, alpha=0.45, edgecolors="none",
                        label=stype.capitalize())
    axes[2].set_xlabel("D_i non-zero fraction\n(cross-feeding output breadth)")
    axes[2].set_ylabel("Mean community abundance")
    axes[2].set_title("Cross-feeding Breadth vs Community Abundance\n"
                      "(positive trend = cross-feeding confers advantage)",
                      fontweight="bold")
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    # Print quick summary to terminal
    n_benefit = int(np.sum(ratio[np.isfinite(ratio)] > 1))
    n_cost    = int(np.sum(ratio[np.isfinite(ratio)] < 1))
    tqdm.write(f"\n    Fig 8 summary: {n_benefit} species benefit from community "
               f"({n_benefit/(n_benefit+n_cost):.1%}), "
               f"{n_cost} are suppressed ({n_cost/(n_benefit+n_cost):.1%})")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig8_solo_vs_community_v5.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    fig8_solo_vs_community_v5.png")

    print("\n  All figures saved.")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    (df, genes, c_matrix, uptake_mask, m, g, broths,
     D_list, species_type, preferred_nutrient,
     pathogen_solo, pairwise_matrix,
     community_N, comm_supp_map) = generate_dataset()

    out_path = f"{OUTPUT_DIR}/micrm_pairwise_dataset_v5.csv"
    df.to_csv(out_path, index=False)
    print(f"\n[Done] Dataset saved → micrm_pairwise_dataset_v5.csv  shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")

    make_figures(df, genes, c_matrix, uptake_mask, m, g, broths,
                 D_list, species_type, preferred_nutrient,
                 pathogen_solo, pairwise_matrix,
                 community_N, comm_supp_map)

    print("\n[Sample rows]:")
    print(df[[
        "species_id", "broth_id", "species_type",
        "dominant_nutrient", "absent_nutrients", "delta_per_h", "maintenance_cost",
        "gene_1", "gene_2", "uptake_c1", "uptake_c2",
        "suppression_score", "community_pathogen_suppression",
    ]].head(10).to_string(index=False))


[1] Generating species traits (sparse uptake) …
    Species counts: {'pathogen': 1, 'specialist': 333, 'generalist': 666}
    Mean active nutrients per species: 12.2  (min=3, max=19)

[2] Building sparse species-specific D_i matrices …
    Mean D_i non-zero fraction: 0.167  (vs 1.000 fully dense)

[3] Generating broth environments (partial membership) …
    Broth_01: dominant=c21, absent=['c3', 'c6', 'c10', 'c20', 'c22'], n_members=171, delta=0.058
    Broth_02: dominant=c12, absent=['c3', 'c7'], n_members=107, delta=0.199
    Broth_03: dominant=c23, absent=['c8', 'c10', 'c18', 'c25'], n_members=135, delta=0.075
    Broth_04: dominant=c24, absent=['c2', 'c9', 'c10', 'c11'], n_members=177, delta=0.112
    Broth_05: dominant=c7, absent=['c3', 'c11', 'c14', 'c26', 'c30'], n_members=167, delta=0.107
    Broth_06: dominant=c20, absent=['c21', 'c28', 'c30'], n_members=107, delta=0.059
    Broth_07: dominant=c16, absent=['c1', 'c11', 'c27'], n_members=117, delta=0.066
    Broth_08: dominant=

  Pathogen solo:   3%|██                                                             | 1/30 [00:00<00:04,  7.10broth/s]

    Broth_01: N_pathogen_solo = 0.9467


  Pathogen solo:   7%|████▏                                                          | 2/30 [00:00<00:05,  5.36broth/s]

    Broth_02: N_pathogen_solo = 2.3438
    Broth_03: N_pathogen_solo = 0.7056


...

    Broth_29: community suppression = 1.000  N_pathogen_comm = 0.0000  n_members=133


  Community ODE: 100%|██████████████████████████████████████████████████████████████| 30/30 [23:54<00:00, 47.83s/broth]


    Broth_30: community suppression = 1.000  N_pathogen_comm = 0.0000  n_members=147

  Dataset shape : (3974, 77)
  suppression_score summary:
count    3974.000
mean        0.255
std         0.242
min         0.000
25%         0.019
50%         0.207
75%         0.422
max         1.000
Name: suppression_score, dtype: float64

  Fraction suppressing pathogen (score > 0): 83.72%

[Done] Dataset saved → micrm_pairwise_dataset_v5.csv  shape: (3974, 77)
  Columns: ['species_id', 'broth_id', 'species_type', 'dominant_nutrient', 'absent_nutrients', 'delta_per_h', 'maintenance_cost', 'gene_1', 'gene_2', 'gene_3', 'gene_4', 'gene_5', 'gene_6', 'gene_7', 'gene_8', 'uptake_c1', 'uptake_c2', 'uptake_c3', 'uptake_c4', 'uptake_c5', 'uptake_c6', 'uptake_c7', 'uptake_c8', 'uptake_c9', 'uptake_c10', 'uptake_c11', 'uptake_c12', 'uptake_c13', 'uptake_c14', 'uptake_c15', 'uptake_c16', 'uptake_c17', 'uptake_c18', 'uptake_c19', 'uptake_c20', 'uptake_c21', 'uptake_c22', 'uptake_c23', 'uptake_c24', 'uptake_c

  Solo ODE (Fig 8): 100%|█████████████████████████████████████████████████████████| 3974/3974 [55:23<00:00,  1.20run/s]



    Fig 8 summary: 0 species benefit from community (0.0%), 986 are suppressed (100.0%)
    fig8_solo_vs_community_v5.png

  All figures saved.

[Sample rows]:
  species_id broth_id species_type dominant_nutrient                  absent_nutrients  delta_per_h  maintenance_cost  gene_1  gene_2  uptake_c1  uptake_c2  suppression_score  community_pathogen_suppression
Species_0003 Broth_01   specialist               c21 ['c3', 'c6', 'c10', 'c20', 'c22']       0.0579            0.4199       1       0     0.0000     0.5449             0.0234                             1.0
Species_0004 Broth_01   specialist               c21 ['c3', 'c6', 'c10', 'c20', 'c22']       0.0579            0.5847       1       1     0.0000     0.0000             0.0000                             1.0
Species_0015 Broth_01   specialist               c21 ['c3', 'c6', 'c10', 'c20', 'c22']       0.0579            0.3882       1       0     0.0616     0.0217             0.0000                             1.0
Species_001